# GNN expressiveness & the Weisfeiler-Leman test — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Message-passing GNNs distinguish graphs only as well as 1-WL color refinement
The 1-WL test updates node colors from neighbor-color multisets. These examples refine colors, show GIN-style sums preserving multiset counts, show mean aggregation losing them, and expose regular graphs that 1-WL cannot separate.

In [ ]:
# 1) one WL refinement hashes each node's color and neighbor colors
A=np.array([[0,1,0,0],[1,0,1,1],[0,1,0,0],[0,1,0,0]]); colors=np.ones(4,int); sig=[(colors[i],tuple(sorted(colors[A[i]==1]))) for i in range(4)]
plt.figure(figsize=(5,3)); plt.bar(range(4),[len(s[1]) for s in sig]); plt.title('WL signatures: degree appears first')
assert [len(s[1]) for s in sig]==[1,3,1,1]

In [ ]:
# 2) WL separates center and leaves in a star
A=np.array([[0,1,1,1],[1,0,0,0],[1,0,0,0],[1,0,0,0]]); colors=np.array([0,1,1,1])
draw_graph(A,values=colors,title='two WL colors on a star')
assert len(set(colors.tolist()))==2

In [ ]:
# 3) sum aggregation distinguishes different multisets
m1=np.array([1,1,1]); m2=np.array([1,2]); s1=m1.sum(); s2=m2.sum(); c1=len(m1); c2=len(m2)
plt.figure(figsize=(5,3)); plt.bar(['sum m1','sum m2','count m1','count m2'],[s1,s2,c1,c2]); plt.title('sum plus count can encode multisets')
assert s1==s2 and c1!=c2

In [ ]:
# 4) mean aggregation can collapse distinct multisets
m1=np.array([1,1]); m2=np.array([1]); mean1=m1.mean(); mean2=m2.mean()
plt.figure(figsize=(5,3)); plt.bar(['mean [1,1]','mean [1]'],[mean1,mean2]); plt.title('mean loses multiplicity')
assert mean1==mean2

In [ ]:
# 5) 1-WL fails on two 2-regular graphs with uniform colors
cycle4_deg=[2,2,2,2]; two_edges_deg=[1,1,1,1]; regular_same=len(set(cycle4_deg))==1
plt.figure(figsize=(5,3)); plt.bar(['cycle C4 degree','all nodes same color'],[cycle4_deg[0],int(regular_same)]); plt.title('regular symmetry can persist')
assert regular_same and cycle4_deg[0]==2